# ✈️ FlightAI: Enterprise Airline Assistant
> **Autonomous Agentic Workflow with Local SQLite Persistence & Gradio UI

## ⚙️ Step 1: Environment Setup & Client Configuration

In [10]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import sqlite3
import gradio as gr

load_dotenv(override=True)

ollama_client = OpenAI(
    base_url = "http://localhost:11434/v1",
    api_key = "ollama"
)
OLLAMA_MODEL = "llama3.2"

gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
GEMINI_MODEL = "gemini-3.5-flash-lite"

## 🗄️ Step 2: Initialize Database & Seed Flight Fares

In [12]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
  cursor = conn.cursor()
  cursor.execute(
      "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
  )

  cites = {"london": 799, "paris": 899, "tokyo": 1420, "berlin": 499}

  for city, price in cites.items():
    cursor.execute(
        "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO"
        " UPDATE SET price = ?",
        (city.lower(), price, price),
    )

  conn.commit()

## 🔍 Step 3: Database Query Function & Tool Schema


In [13]:
def get_ticket_price(destination_city): 
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT price FROM prices WHERE city = ?",
            (destination_city.lower().strip(),),
        )
        result = cursor.fetchone()

    if result:
        return f"The ticket price to {destination_city.capitalize()} is ${result[0]:.2f}."
    else:
        return (
            f"No ticket price information found for"
            f" '{destination_city.capitalize()}'."
        )
    
price_function = {
    "name": "get_ticket_price",
    "description": (
        "Retrieve the deterministic flight ticket price for a given destination"
        " city from the local database."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": (
                    "The name of the target arrival city (e.g. london, paris,"
                    " tokyo)."
                ),
            }
        },
        "required": ["destination_city"],
    },
}

tools = [{"type": "function", "function": price_function}]



## 🛠️ Step 4: Tool Call Handler & Output Cleaner


In [14]:
KNOWN_CITIES = ["london", "paris", "tokyo", "berlin"]


def clean_output(text):
  cleaned = text.strip()
  if cleaned.startswith("assistant"):
    cleaned = cleaned[len("assistant") :].strip()
  return cleaned


def handle_tool_call(message):
  tool_call = message.tool_calls[0]
  if tool_call.function.name == "get_ticket_price":
    try:
      arguments = json.loads(tool_call.function.arguments)
    except Exception:
      arguments = {}

    city = arguments.get("destination_city", "")
    price_details = get_ticket_price(city)
    return {
        "role": "tool",
        "content": price_details,
        "tool_call_id": tool_call.id,
    }

## 🧠 Step 5: Conversation Logic & Multi-City Handling



In [25]:
system_message = (
    "You are FlightAI, a polite and helpful airline customer support assistant."
    " Respond politely to greetings. When asked about flight prices, always"
    " provide the exact retrieved prices clearly without guessing."
)


def chat(message, history, model_choice):
  
  if model_choice == "Gemini":
    client = gemini_client
    model = GEMINI_MODEL
  else:
    client = ollama_client
    model = OLLAMA_MODEL

  user_msg_lower = message.lower()

  formatted_history = []
  for h in history:
    if isinstance(h, dict):
      formatted_history.append({"role": h["role"], "content": h["content"]})
    elif isinstance(h, (list, tuple)) and len(h) == 2:
      formatted_history.append({"role": "user", "content": h[0]})
      formatted_history.append({"role": "assistant", "content": h[1]})

  messages = (
      [{"role": "system", "content": system_message}]
      + formatted_history
      + [{"role": "user", "content": message}]
  )

  detected_cities = [c for c in KNOWN_CITIES if c in user_msg_lower]
  if detected_cities:
    prices = [get_ticket_price(city) for city in detected_cities]
    messages.append({
        "role": "system",
        "content": (
            "Database results:\n"
            + "\n".join(prices)
            + "\nState these exact prices clearly to the user without making up"
            " different numbers."
        ),
    })
    res = client.chat.completions.create(model=model, messages=messages)
    return clean_output(res.choices[0].message.content)

  # General conversation or model-triggered tool calling
  response = client.chat.completions.create(
      model=model, messages=messages, tools=tools
  )

  while response.choices[0].finish_reason == "tool_calls":
    msg = response.choices[0].message
    tool_resp = handle_tool_call(msg)
    messages.append(msg)
    messages.append(tool_resp)
    response = client.chat.completions.create(
        model=model, messages=messages, tools=tools
    )

  return clean_output(response.choices[0].message.content)


## 🚀 Step 6: Launching the Gradio Web Interface


In [ ]:
with gr.Blocks(title="FlightAI Assistant") as demo:
  gr.Markdown("## ✈️ FlightAI Assistant")
  gr.Markdown(
      "Ask about flight ticket prices (e.g., London, Paris, Tokyo, Berlin)."
  )

  model_choice = gr.Radio(
      choices=["Gemini", "Ollama"],
      value="Gemini",
      label="Choose the model you want",
      interactive=True,
  )

  gr.ChatInterface(
      fn=chat,
      additional_inputs=[model_choice],
  )

demo.launch(inbrowser=True)